### Criar Testes Unitários Simples em um Notebook Databricks

Criar um teste unitário para uma função:

#### Criar a Função Inicial: 

Com entradas (inputs) e saídas (outputs) que possam ser validadas.

#### Criar Funções de Teste: 

Escrever uma função de teste unitário que chame a função inicial e verifique se a saída real da função corresponde ao resultado esperado definido usando asserções (assertions). 

#### Executar as Funções de Teste: 

Executar e revisar o teste usando um framework de testes Pytest. 

Revisar os resultados validando que a função se comportou como esperado e corrija quaisquer problemas se o teste falhar.


#### Importar as funcoes do project_functions da pasta Helpers

In [0]:
import sys 
import os

# Obtém o diretório atual 
current_path = os.getcwd()

#"/Workspace/Users/carlosdfrota@gmail.com/DataBricks-Courses/DataBricks - Data Engineer Learning Plan/Lab - Pyspark Testing/Helpers"

# Adiciona o caminho da pasta raiz
root_folder_path = os.path.dirname(os.path.dirname(current_path))

# Ao adicionar root_folder_path, permitindo o Notebook importar qualquer coisa que esteja dentro da estrutura do seu projeto.
sys.path.append(root_folder_path)

print(f'Pasta:{root_folder_path} adicionada ao projeto ')

#### OBS
Não é possivel importar notebooks usando import o arquivo deve ser um arquivo do tupo Python .py

In [0]:
# O import precisa refletir o caminho a partir da raiz
from Helpers import project_functions

#dbutils.import_notebook("Helpers.project_functions")

### Criação de 2 funções de teste 


#### test_get_health_csv_schema_match 

•	A variável actual_schema armazena o schema retornado pela função get_health_csv_schema. 

•	A variável expected_schema especifica o schema que esperamos que a função retorne se estiver funcionando corretamente. 

•	A instrução assertSchemaEqual de pyspark.testing.utils compara o expected_schema com o actual_schema.

Se forem iguais → o teste passa. 

Se forem diferentes → o teste falha, indicando problema na função.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, DoubleType, DateType
from pyspark.sql.functions import when, col

# Importa Funcao assertSchemaEqual da Bliblioteca de Testes do PySpark
from pyspark.testing.utils import assertDataFrameEqual, assertSchemaEqual

In [0]:
def test_get_health_csv_schema_match ():
    # Obtém o schema da nossa função
    actual_schema = project_functions.get_health_csv_schema()

    # Definir o schema esperado
    expected_schema = StructType([
        StructField("id", IntegerType(), True),
        StructField("PII", StringType(), True), 
        StructField("date", DateType(), True),
        StructField("HighCholest", IntegerType(), True),
        StructField("HighBP", DoubleType(), True), 
        StructField("BMI", DoubleType(), True),
        StructField("Age", DoubleType(), True), 
        StructField("Education", DoubleType(), True),
        StructField("income", IntegerType(), True)
    ])

    # Verifica se o schema atual corresponde ao esperado
    assertSchemaEqual(actual_schema,expected_schema)
    print('Teste executado com sucesso')

test_get_health_csv_schema_match()

#### test_high_cholest_column_valid_map 

Importa a função assertDataFrameEqual do módulo pyspark.testing.utils para comparar dois DataFrames PySpark e garantir que sejam iguais, verificando tanto os dados quanto o schema. 

•	A variável actual_df armazena o novo DataFrame criado a partir do resultado da função highcholest_map. 

•	A variável expected_df contém o DataFrame esperado que antecipamos como resultado da função. 

•	A função assertDataFrameEqual de pyspark.testing.utils compara os DataFrames actual_df e expected_df.

Se os dados ou o schema não forem iguais, um erro será gerado.


In [0]:
def test_high_cholest_column_valid_map():
    data = [
        (0,),
        (1,),
        (2,),
        (3,),
        (4,),
        (None,)
    ]
    sample_df = spark.createDataFrame(data,["value"])

    # Aplicar funcao nos dados de entrada
    actual_df = sample_df.withColumn("actual", project_functions.high_cholest_map("value"))

    # Criar DataFrame estático com os resultados esperados da função highcholest_map acima:
    expected_df = spark.createDataFrame(
        [
            (0,'Normal'),
            #(0, "Bad Value Cause Error"),	### <-- change the value to cause an error        
            (1,'Acima da Media'),
            (2, "Alto"),
            (3, "Desconhecido"),
            (4, "Desconhecido"),
            (None, "Desconhecido")
        ],
        schema = StructType(
            [
                StructField('value',LongType(),True),
                StructField('actual',StringType(),False),
            ])
    )
    # Checar se os resultados são iguais. Se não retorna erro
    assertDataFrameEqual(actual_df,expected_df)
    print("Test passed!")
    #return actual_df  # ← adicione o return


In [0]:
# Validar se o actuak_df esta sendo carregado corretamente
# foi necessario incluir o return na funcao
# e apos adicionar o resultado da funcao em uma variavel antes de exibila

teste = test_high_cholest_column_valid_map()
display(teste)

In [0]:
test_high_cholest_column_valid_map()